In [1]:
import csv, json, os, shutil, hashlib
from pathlib import Path

# ================= STUDENT PARAMETERS =================
STUDENT_ID   = "TUPM-26-1859"
DIGITS       = [int(c) for c in STUDENT_ID if c.isdigit()]
ID_SUM       = sum(DIGITS)                          # 31
SEED_DIGIT   = DIGITS[-1]                           # 9
TARGET_OS    = "Android" if SEED_DIGIT % 2 == 0 else "iOS"   # 9 is odd -> iOS
TARGET_CLASS = (SEED_DIGIT % 5) + 1                 # (9 % 5) + 1 = 5

# ================= LOCATE DATASET =================
FILE_NAME   = "user_behavior_dataset.csv"
FOLDER_NAME = "Lab6-Additional-Exercises"
MANUAL_PATH = r""    # optional: paste full path to the CSV here

def find_dataset():
    if MANUAL_PATH and Path(MANUAL_PATH).is_file():
        return Path(MANUAL_PATH)
    cwd = Path.cwd()
    for p in (cwd / FILE_NAME, cwd / FOLDER_NAME / FILE_NAME):
        if p.is_file():
            return p
    for base in [cwd, *cwd.parents, Path.home()]:
        for hit in base.rglob(FILE_NAME):
            if ".venv" not in hit.parts and "output" not in hit.parts and "backup" not in hit.parts and hit.is_file():
                return hit
    return None

SRC = find_dataset()
if SRC is None:
    raise FileNotFoundError(f"{FILE_NAME} not found. Working dir: {Path.cwd()}\nFiles: {os.listdir()}")
SRC = SRC.resolve()
PROJECT = SRC.parent

OUT_DIR, BAK_DIR = PROJECT / "output", PROJECT / "backup"
OUT_DIR.mkdir(exist_ok=True); BAK_DIR.mkdir(exist_ok=True)
tag = f"{TARGET_OS.lower()}_class{TARGET_CLASS}"
EXPORT_CSV   = OUT_DIR / f"mobile_{tag}_users.csv"
SUMMARY_JSON = OUT_DIR / f"mobile_{tag}_summary.json"
BACKUP_CSV   = BAK_DIR / f"mobile_{tag}_users_backup.csv"

# ================= READ & FILTER =================
def col(header, *names):
    lk = {h.strip().lower(): h for h in header}
    for n in names:
        if n.lower() in lk:
            return lk[n.lower()]
    raise KeyError(f"Missing column {names} in {header}")

total = os_match = invalid = 0
selected = []          # only the selected users (small), exported later
battery_sum = data_sum = 0.0

with open(SRC, newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    c_id  = col(reader.fieldnames, "User ID")
    c_dev = col(reader.fieldnames, "Device Model")
    c_os  = col(reader.fieldnames, "Operating System")
    c_bat = col(reader.fieldnames, "Battery Drain (mAh/day)")
    c_dat = col(reader.fieldnames, "Data Usage (MB/day)")
    c_cls = col(reader.fieldnames, "User Behavior Class")

    for row in reader:
        total += 1
        os_name = (row.get(c_os) or "").strip()
        if os_name.lower() != TARGET_OS.lower():
            continue
        os_match += 1
        try:
            cls = int(float((row.get(c_cls) or "").strip()))
            bat = float((row.get(c_bat) or "").strip())
            dat = float((row.get(c_dat) or "").strip())
            uid = (row.get(c_id) or "").strip()
            dev = (row.get(c_dev) or "").strip()
            if not uid or not dev:
                raise ValueError
        except (ValueError, TypeError):
            invalid += 1
            continue
        if cls == TARGET_CLASS:
            selected.append([uid, dev, os_name, bat, dat])
            battery_sum += bat
            data_sum += dat

n_sel = len(selected)
avg_bat = round(battery_sum / n_sel, 2) if n_sel else None
avg_dat = round(data_sum / n_sel, 2) if n_sel else None

# ================= EXPORT CSV =================
with open(EXPORT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["User ID", "Device Model", "Operating System", "Battery Drain (mAh/day)", "Data Usage (MB/day)"])
    w.writerows(selected)

# ================= VERIFY COUNT =================
with open(EXPORT_CSV, newline="", encoding="utf-8") as f:
    exported_rows = sum(1 for _ in csv.DictReader(f))
count_ok = exported_rows == n_sel

# ================= JSON SUMMARY =================
summary = {
    "student_id": STUDENT_ID, "source_file": str(SRC),
    "target_operating_system": TARGET_OS, "target_behavior_class": TARGET_CLASS,
    "total_records_in_dataset": total, "records_matching_os": os_match,
    "invalid_records_skipped": invalid, "records_in_target_class": n_sel,
    "average_battery_drain_mAh_per_day": avg_bat, "average_data_usage_MB_per_day": avg_dat,
    "exported_row_count": exported_rows, "count_verified": count_ok,
    "exported_csv_file": str(EXPORT_CSV),
    "note": None if n_sel else "No records matched the OS and behavior class.",
}
with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

# ================= BACKUP + SIZE =================
shutil.copy2(EXPORT_CSV, BACKUP_CSV)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
backup_ok = BACKUP_CSV.exists() and sha(EXPORT_CSV) == sha(BACKUP_CSV)
backup_size = os.path.getsize(BACKUP_CSV)

# ================= ASSESSMENT DATA =================
fmt = lambda v, u: f"{v:.2f} {u}" if v is not None else "N/A (no matching records)"
print("=" * 70)
print(f"ASSESSMENT DATA  |  {STUDENT_ID}  (ID_SUM={ID_SUM}, SEED_DIGIT={SEED_DIGIT})")
print("=" * 70)
print(f"Source file                        : {SRC}")
print(f"Operating system / behavior class  : {TARGET_OS} / Class {TARGET_CLASS}")
print(f"Total records in dataset           : {total}")
print(f"Records matching operating system  : {os_match}  (invalid/skipped: {invalid})")
print(f"Records in target behavior class   : {n_sel}")
print(f"Average battery drain (mAh/day)    : {fmt(avg_bat, 'mAh/day')}")
print(f"Average data usage (MB/day)        : {fmt(avg_dat, 'MB/day')}")
print(f"Count verification                 : {'PASSED' if count_ok else 'FAILED'} (exported {exported_rows} vs selected {n_sel})")
print(f"Exported CSV file                  : {EXPORT_CSV}")
print(f"JSON summary file                  : {SUMMARY_JSON}")
print(f"Backup file and size               : {BACKUP_CSV}  ({backup_size:,} bytes, {backup_size/1024:.2f} KB)  [{'VERIFIED' if backup_ok else 'MISMATCH'}]")
print("=" * 70)

# ================= FINAL OUTPUT =================
print("\nFinal Output:")
print("-" * 70)
print(f"Operating system and behavior class : {TARGET_OS}, Class {TARGET_CLASS}")
print(f"Total records in dataset            : {total}")
print(f"Records matching operating system   : {os_match}")
print(f"Records in target behavior class    : {n_sel}")
print(f"Average battery drain (mAh/day)     : {avg_bat if avg_bat is not None else 'N/A'}")
print(f"Average data usage (MB/day)         : {avg_dat if avg_dat is not None else 'N/A'}")
print(f"Count verification                  : {'PASSED' if count_ok else 'FAILED'}")
print(f"Exported CSV file                   : {EXPORT_CSV.name}")
print(f"JSON summary file                   : {SUMMARY_JSON.name}")
print(f"Backup file and size                : {BACKUP_CSV.name} ({backup_size:,} bytes)")
print("-" * 70)
if n_sel:
    print(f"RESULT: {n_sel} of {os_match} {TARGET_OS} users belong to behavior class {TARGET_CLASS}, "
          f"averaging {avg_bat} mAh/day battery drain and {avg_dat} MB/day data usage.")
else:
    print(f"RESULT: No {TARGET_OS} users found in behavior class {TARGET_CLASS}.")

ASSESSMENT DATA  |  TUPM-26-1859  (ID_SUM=31, SEED_DIGIT=9)
Source file                        : C:\Users\jullm\Lab6_File_Handling_Pacudan\Lab6-Additional-Exercises\user_behavior_dataset.csv
Operating system / behavior class  : iOS / Class 5
Total records in dataset           : 700
Records matching operating system  : 146  (invalid/skipped: 0)
Records in target behavior class   : 31
Average battery drain (mAh/day)    : 2727.97 mAh/day
Average data usage (MB/day)        : 1940.90 MB/day
Count verification                 : PASSED (exported 31 vs selected 31)
Exported CSV file                  : C:\Users\jullm\Lab6_File_Handling_Pacudan\Lab6-Additional-Exercises\output\mobile_ios_class5_users.csv
JSON summary file                  : C:\Users\jullm\Lab6_File_Handling_Pacudan\Lab6-Additional-Exercises\output\mobile_ios_class5_summary.json
Backup file and size               : C:\Users\jullm\Lab6_File_Handling_Pacudan\Lab6-Additional-Exercises\backup\mobile_ios_class5_users_backup.csv  (1,10